<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module3/m3_l3_nb1_denoising.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧹 Module 3 · Lesson 3 · Notebook 1
# Building a Denoising Autoencoder

---

**Module 3 · Introduction to Autoencoders**  
**Estimated time:** 35 minutes  
**Prerequisites:** Lesson 2 notebooks complete — you should be comfortable building and training a basic autoencoder on MNIST

---

## 📋 What This Notebook Does

In Lesson 2 you built an autoencoder that reconstructs its own clean input. This notebook makes one conceptual change that transforms it into something more powerful: the network receives a **corrupted** version of the input and is asked to produce the **clean** original.

That single change — noisy input, clean target — forces the network to learn the underlying structure of the data rather than its surface-level pixel statistics. The result is a model that can genuinely remove noise from images it has never seen before.

By the end of this notebook you will have:

- Added Gaussian noise to MNIST images and visualised the effect
- Trained a denoising autoencoder with the corrupted-input / clean-target objective
- Produced a three-row comparison: noisy input / denoised output / clean original
- Investigated how noise level affects the model's ability to recover the clean signal
- Compared the representations learned by a standard vs denoising autoencoder
- Written an exit reflection connecting autoencoders to your own domain

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Imports, seeds, load and preprocess MNIST |
| **1. Add Noise** | Generate noisy training and test images |
| **2. Visualise Noise** | See how different noise levels affect the images |
| **3. Build the Model** | Same architecture as Lesson 2 |
| **4. Train** | One changed line — noisy input, clean target |
| **5. Visualise Results** | Three-row comparison grid |
| **6. Noise Level Experiment** | Investigate the model's limits |
| **7. Compare Representations** | Standard vs denoising latent space |
| **8. Exit Reflection** | Connect to your domain |

---

> **The key idea to hold in mind throughout:** The architecture does not change — only the training objective does. The same encoder–bottleneck–decoder structure, trained on a different task, learns fundamentally different and more robust internal representations.

---
## Step 0 — Setup

Same imports and preprocessing as Lesson 2. If you are continuing in the same Colab session you can skip this cell — your data is already in memory.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

np.random.seed(42)
tf.random.set_seed(42)

# Load and preprocess MNIST — identical to Lesson 2
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.mnist.load_data()

x_train = x_train_raw.astype('float32') / 255.0
x_test  = x_test_raw.astype('float32')  / 255.0
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

print(f'TensorFlow: {tf.__version__}')
print(f'Training set: {x_train.shape}  |  Test set: {x_test.shape}')
print('✅ Setup complete.')

---
## Step 1 — Add Noise to the Images

### 1.1 — Gaussian Noise

We add random values drawn from a Gaussian (normal) distribution to each pixel. The `noise_factor` controls the standard deviation of the noise — higher values produce more severe corruption.

**Why clip to [0, 1]?**  
After adding noise, some pixel values may exceed 1.0 or fall below 0.0. We clip them back to the valid range because:
- Our normalised pixels live in [0, 1]
- Binary cross-entropy requires inputs in (0, 1)
- Values outside this range are physically meaningless as pixel intensities

> **Critical detail:** We add different random noise to each image at training time. Because the noise is never the same twice, the network cannot memorise it. The only reliable strategy for producing clean outputs is to learn the underlying structure of the digits.

In [ ]:
# ── Noise parameter — change this to experiment ──
NOISE_FACTOR = 0.3

def add_noise(images, noise_factor, seed=None):
    """Add Gaussian noise and clip to valid range."""
    if seed is not None:
        np.random.seed(seed)
    noise   = noise_factor * np.random.normal(size=images.shape)
    noisy   = images + noise
    return np.clip(noisy, 0.0, 1.0).astype('float32')

x_train_noisy = add_noise(x_train, NOISE_FACTOR, seed=42)
x_test_noisy  = add_noise(x_test,  NOISE_FACTOR, seed=99)  # different seed = different noise pattern

print(f'Noise factor: {NOISE_FACTOR}')
print(f'Noisy train: {x_train_noisy.shape}  value range [{x_train_noisy.min():.2f}, {x_train_noisy.max():.2f}]')
print(f'Noisy test:  {x_test_noisy.shape}')
print()
print('The training input  = noisy images   (x_train_noisy)')
print('The training target = clean originals (x_train)')
print('This is the only structural difference from a standard autoencoder.')

---
## Step 2 — Visualise the Noise

Before training, it helps to see what your model will actually receive as input. The cell below shows original and noisy versions side by side for several noise levels so you can build intuition for what `NOISE_FACTOR` means visually.

In [ ]:
noise_levels = [0.1, 0.3, 0.5, 0.7, 1.0]
n_examples   = 5

fig, axes = plt.subplots(len(noise_levels) + 1, n_examples, figsize=(11, 14))

# Row 0: clean originals
for col in range(n_examples):
    axes[0, col].imshow(x_test[col].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[0, col].axis('off')
axes[0, 0].set_ylabel('Clean\n(original)', fontsize=8, rotation=0, labelpad=50, va='center')

# Rows 1+: noisy versions at each level
for row, nf in enumerate(noise_levels):
    noisy_sample = add_noise(x_test[:n_examples], nf)
    for col in range(n_examples):
        axes[row+1, col].imshow(noisy_sample[col].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[row+1, col].axis('off')
    axes[row+1, 0].set_ylabel(f'σ = {nf}', fontsize=8, rotation=0, labelpad=50, va='center')

plt.suptitle('Effect of Gaussian Noise at Different Levels\n'
             f'Current NOISE_FACTOR = {NOISE_FACTOR} (highlighted row)',
             fontsize=11, y=1.01)

# Highlight the row matching NOISE_FACTOR
if NOISE_FACTOR in noise_levels:
    row_idx = noise_levels.index(NOISE_FACTOR) + 1
    for col in range(n_examples):
        axes[row_idx, col].set_facecolor('#fffbe6')
        for spine in axes[row_idx, col].spines.values():
            spine.set_edgecolor('#f1c40f')
            spine.set_linewidth(2)

plt.tight_layout()
plt.show()

print(f'At NOISE_FACTOR = {NOISE_FACTOR}, the digits are still recognisable but clearly degraded.')
print('The model must learn to recover the clean structure from this corrupted input.')

---
## Step 3 — Build the Model

**The architecture is identical to Lesson 2.** This is intentional — it demonstrates that what the network learns is determined by the training objective, not the architecture.

```
Encoder:  784 → Dense(128, ReLU) → Dense(32)        ← bottleneck
Decoder:  32  → Dense(128, ReLU) → Dense(784, σ)    ← sigmoid output
```

The only difference from Lesson 2 will come in Step 4, in the training call.

In [ ]:
LATENT_DIM = 32

# ── Encoder ──
enc_in  = keras.Input(shape=(784,), name='enc_input')
enc_h   = layers.Dense(128, activation='relu', name='enc_hidden')(enc_in)
enc_out = layers.Dense(LATENT_DIM, activation=None, name='latent')(enc_h)
encoder = Model(enc_in, enc_out, name='encoder')

# ── Decoder ──
dec_in  = keras.Input(shape=(LATENT_DIM,), name='dec_input')
dec_h   = layers.Dense(128, activation='relu', name='dec_hidden')(dec_in)
dec_out = layers.Dense(784, activation='sigmoid', name='dec_output')(dec_h)
decoder = Model(dec_in, dec_out, name='decoder')

# ── Full autoencoder ──
ae_input  = keras.Input(shape=(784,))
ae_output = decoder(encoder(ae_input))
autoencoder = Model(ae_input, ae_output, name='denoising_autoencoder')

autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

print('Denoising autoencoder built — identical architecture to Lesson 2.')
print(f'Latent dimension: {LATENT_DIM}')
print(f'Total parameters: {autoencoder.count_params():,}')
print()
print('The only change from Lesson 2 comes in the next step: what we pass as input vs target.')

---
## Step 4 — Train the Denoising Autoencoder

### The One Change

Compare these two training calls carefully:

```python
# Standard autoencoder (Lesson 2)
autoencoder.fit(x_train, x_train, ...)        # input = clean, target = clean

# Denoising autoencoder (this notebook)
autoencoder.fit(x_train_noisy, x_train, ...)  # input = noisy, target = clean
```

That is the entirety of the structural difference. The loss function still measures the difference between the decoder output and the target — but now the target is the clean original, not the noisy input. The network is penalised for any noise it fails to remove.

In [ ]:
print('Training the denoising autoencoder...')
print(f'  Input:  x_train_noisy  (noise_factor = {NOISE_FACTOR})')
print(f'  Target: x_train        (clean originals)')
print()

history = autoencoder.fit(
    x_train_noisy, x_train,    # ← THE KEY CHANGE: noisy input, clean target
    epochs=20,
    batch_size=256,
    shuffle=True,
    validation_split=0.1,
    verbose=1
)

print('\n✅ Training complete.')
print(f'Final training loss:   {history.history["loss"][-1]:.5f}')
print(f'Final validation loss: {history.history["val_loss"][-1]:.5f}')

In [ ]:
# Loss curves
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.history['loss'],     label='Training loss',   color='steelblue', linewidth=2)
ax.plot(history.history['val_loss'], label='Validation loss', color='tomato',    linewidth=2, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Binary Cross-Entropy Loss')
ax.set_title(f'Denoising Autoencoder — Training Curves  (noise_factor = {NOISE_FACTOR})',
             fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Step 5 — Visualise the Results

The standard evaluation for a denoising autoencoder is a **three-row comparison**:

- **Row 1:** Noisy input — what the network receives
- **Row 2:** Denoised output — what the network produces
- **Row 3:** Clean original — the ground truth target

This layout makes it immediately clear how much noise is being removed and how close the output is to the true clean image.

> **What to look for:** Are the denoised images clearly cleaner than the noisy inputs? How close are they to the originals? Which digits denoise best? Which are most distorted?

In [ ]:
# Generate denoised outputs on test set
denoised = autoencoder.predict(x_test_noisy, verbose=0)

n_display = 10
fig, axes = plt.subplots(3, n_display, figsize=(16, 5.5))

row_labels = ['Noisy Input', 'Denoised', 'Clean Original']
row_data   = [x_test_noisy, denoised, x_test]

for row_idx, (label, data) in enumerate(zip(row_labels, row_data)):
    for col in range(n_display):
        axes[row_idx, col].imshow(data[col].reshape(28, 28),
                                   cmap='gray', vmin=0, vmax=1)
        axes[row_idx, col].axis('off')
        if col == 0:
            axes[row_idx, col].set_ylabel(label, fontsize=8,
                                           rotation=0, labelpad=65, va='center')

plt.suptitle(f'Denoising Autoencoder Results  —  noise_factor = {NOISE_FACTOR}',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Quantitative comparison
mse_noisy    = np.mean((x_test - x_test_noisy) ** 2)
mse_denoised = np.mean((x_test - denoised)     ** 2)
improvement  = (mse_noisy - mse_denoised) / mse_noisy * 100

print(f'Quantitative noise reduction:')
print(f'  MSE (noisy vs clean):    {mse_noisy:.5f}')
print(f'  MSE (denoised vs clean): {mse_denoised:.5f}')
print(f'  Improvement:             {improvement:.1f}% reduction in reconstruction error')

### 5.1 — Per-Digit Denoising Quality

In [ ]:
print('Average denoising MSE per digit class (lower = better denoising):')
print(f'{"Digit":>8}  {"Noisy→Clean MSE":>18}  {"Denoised→Clean MSE":>20}  {"Improvement":>12}')
print('-' * 65)

for digit in range(10):
    mask = y_test == digit
    mse_n = np.mean((x_test[mask] - x_test_noisy[mask]) ** 2)
    mse_d = np.mean((x_test[mask] - denoised[mask])     ** 2)
    impv  = (mse_n - mse_d) / mse_n * 100
    bar   = '█' * int(impv / 3)
    print(f'{digit:>8}  {mse_n:>18.5f}  {mse_d:>20.5f}  {impv:>10.1f}%  {bar}')

---
## Step 6 — Noise Level Experiment

How well does the model hold up as noise increases? The reading page for this lesson asked: *"At what level of noise does the model start to struggle?"*

The cell below evaluates the trained model (trained at `NOISE_FACTOR = 0.3`) against test images corrupted at different noise levels. This shows both the model's operating range and its graceful degradation under severe noise.

In [ ]:
test_noise_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]
n_show = 5

fig, axes = plt.subplots(len(test_noise_levels) * 2, n_show, figsize=(10, 20))

mse_results = []
for i, nf in enumerate(test_noise_levels):
    noisy_at_nf   = add_noise(x_test[:n_show], nf)
    denoised_at_nf = autoencoder.predict(noisy_at_nf, verbose=0)
    mse_at_nf      = np.mean((x_test[:n_show] - denoised_at_nf) ** 2)
    mse_results.append(mse_at_nf)

    for col in range(n_show):
        axes[i*2, col].imshow(noisy_at_nf[col].reshape(28,28), cmap='gray', vmin=0, vmax=1)
        axes[i*2, col].axis('off')
        axes[i*2+1, col].imshow(denoised_at_nf[col].reshape(28,28), cmap='gray', vmin=0, vmax=1)
        axes[i*2+1, col].axis('off')

    trained_marker = ' ← trained here' if nf == NOISE_FACTOR else ''
    axes[i*2,   0].set_ylabel(f'σ={nf} noisy{trained_marker}',   fontsize=7, rotation=0, labelpad=75, va='center')
    axes[i*2+1, 0].set_ylabel(f'σ={nf} denoised',               fontsize=7, rotation=0, labelpad=75, va='center')

plt.suptitle(f'Model trained at σ={NOISE_FACTOR} — evaluated across noise levels',
             fontsize=11, y=1.005)
plt.tight_layout()
plt.show()

# Plot MSE vs noise level
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(test_noise_levels, mse_results, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.axvline(NOISE_FACTOR, color='tomato', linestyle='--', linewidth=1.5,
           label=f'Training noise level (σ={NOISE_FACTOR})')
ax.set_xlabel('Test Noise Level (σ)', fontsize=12)
ax.set_ylabel('Denoised vs Clean MSE', fontsize=12)
ax.set_title('Denoising Quality vs Noise Level', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('📊 Notice: the model performs best near the noise level it was trained on.')
print('   At higher noise levels, quality degrades — the model is extrapolating')
print('   beyond what it learned during training.')

---
## Step 7 — Compare Representations: Standard vs Denoising

The reading made a strong claim: *"The latent representations learned by a denoising autoencoder are often more useful for downstream tasks than those learned by a standard autoencoder."* Let us test this directly.

We train a standard autoencoder on clean MNIST, then compare the 2D latent space scatter plots for both models. The denoising model should show tighter, better-separated clusters — evidence that its representations are more structured and semantically organised.

In [ ]:
# Build and train a standard autoencoder (2D bottleneck for scatter plot comparison)
def build_ae_2d(name):
    ei = keras.Input(shape=(784,))
    eh = layers.Dense(128, activation='relu')(ei)
    eo = layers.Dense(2,   activation=None)(eh)
    enc = Model(ei, eo, name=f'enc_{name}')

    di  = keras.Input(shape=(2,))
    dh  = layers.Dense(128, activation='relu')(di)
    do_ = layers.Dense(784, activation='sigmoid')(dh)
    dec = Model(di, do_, name=f'dec_{name}')

    ai  = keras.Input(shape=(784,))
    ao  = dec(enc(ai))
    ae  = Model(ai, ao, name=f'ae_{name}')
    ae.compile(optimizer='adam', loss='binary_crossentropy')
    return enc, ae

enc_std, ae_std = build_ae_2d('standard')
enc_den, ae_den = build_ae_2d('denoising')

print('Training standard autoencoder (2D bottleneck)...')
ae_std.fit(x_train, x_train,
           epochs=20, batch_size=256, shuffle=True,
           validation_split=0.1, verbose=0)

print('Training denoising autoencoder (2D bottleneck)...')
ae_den.fit(x_train_noisy, x_train,
           epochs=20, batch_size=256, shuffle=True,
           validation_split=0.1, verbose=0)

print('✅ Both models trained. Generating latent space plots...')

# Extract latent vectors
z_std = enc_std.predict(x_test, verbose=0)
z_den = enc_den.predict(x_test, verbose=0)

# Side-by-side scatter plots
cmap   = plt.get_cmap('tab10')
colors = [cmap(i) for i in range(10)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for digit in range(10):
    mask = y_test == digit
    ax1.scatter(z_std[mask,0], z_std[mask,1], c=[colors[digit]],
                label=str(digit), alpha=0.3, s=6, edgecolors='none')
    ax2.scatter(z_den[mask,0], z_den[mask,1], c=[colors[digit]],
                label=str(digit), alpha=0.3, s=6, edgecolors='none')

ax1.set_title('Standard Autoencoder\n(trained on clean input)', fontsize=12, fontweight='bold')
ax2.set_title(f'Denoising Autoencoder\n(trained with noise_factor = {NOISE_FACTOR})',
              fontsize=12, fontweight='bold')

for ax in [ax1, ax2]:
    ax.set_xlabel('Latent Dimension 1'); ax.set_ylabel('Latent Dimension 2')
    ax.legend(title='Digit', fontsize=8, markerscale=3, loc='upper right')
    ax.grid(alpha=0.2)

plt.suptitle('Latent Space Comparison — Standard vs Denoising Autoencoder',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 Compare the cluster separation between the two plots.')
print('   Does the denoising model show tighter, more separated clusters?')
print('   This is evidence that its representations are more structurally organised.')

---
## Step 8 — Exit Reflection

These two questions are the exit reflection for Lesson 3 and Module 3. Write 2–3 sentences for each. There are no right or wrong answers — the goal is honest engagement with what you have learned.

---

### ❓ Reflection 1 — What resonated?

**What is one thing about autoencoders — from any part of this module — that you found most interesting or surprising? It could be a technical idea, a conceptual insight, or a result you did not expect to see.**

*Some possibilities to prompt your thinking: the way the bottleneck forces meaningful compression without any labels; the fact that adding noise during training produces better representations than training on clean data; the emergence of digit clustering in the latent space with no supervision; the way one training-call change transforms the entire learning objective.*

> **Your answer:** *(Replace this text)*

---

### ❓ Reflection 2 — Your domain

**Where do you think autoencoders could be useful in a field or problem you care about? What would "noisy" and "clean" mean in that context? What would count as an anomaly worth detecting?**

*Think about the data you work with or are most interested in — images, time series, spectra, biological sequences, financial data, sensor readings. What compression, denoising, or anomaly detection problem would be genuinely useful to solve?*

> **Your answer:** *(Replace this text)*

---

## ✅ Notebook 1 Complete

| ✅ | Achievement |
|----|-------------|
| ✅ | Added Gaussian noise to MNIST and visualised the effect across noise levels |
| ✅ | Trained a denoising autoencoder with the corrupted-input / clean-target objective |
| ✅ | Produced a three-row comparison: noisy / denoised / clean original |
| ✅ | Quantified noise reduction per digit class |
| ✅ | Investigated how performance changes at noise levels above the training level |
| ✅ | Compared standard vs denoising latent space scatter plots |
| ✅ | Completed the module exit reflection |

---

### ➡️ Optional: Notebook 2 — Anomaly Detection

If you have time and want to go further, Notebook 2 applies the same reconstruction error principle to anomaly detection — using a trained autoencoder to flag data points that do not fit the patterns it has learned, without ever having seen an anomaly during training.